# 03 — Gold Feature Engineering

Build the **driver-race feature mart** from Silver Parquet using **PySpark** + **DuckDB** validation.

- **Engine:** `GOLD_ENGINE = "spark"`. Spark is the official engine; pandas fallback is manual only (`allow_fallback=True`).
- **Grain:** one row per `session_key`, `meeting_key`, `driver_number`
- **Base:** `session_result_clean.parquet`
- **Target:** `points_finish` = 1 if `points` > 0, else 0
- **No modeling** in this notebook — features and leakage guard only

Run after `02_silver_cleaning_quality.ipynb` with the same `USE_GOOGLE_DRIVE` setting.


## Connection to grading rubric

| Rubric area | This notebook |
|-------------|----------------|
| Gold layer | Feature mart at `data/gold/driver_race_feature_mart.parquet` |
| Target variable | `points_finish` from session points |
| Feature engineering | Lap, pit, position, weather, race control, metadata |
| Data quality | Gold DQ reports under `reports/data_quality/` |
| Leakage prevention | `gold_leakage_guard_report.csv` + feature dictionary |


## Colab setup (required every session)

Identical to `00`–`02`: clone, `pip install -e .`, Drive mount, set `OPENF1_DATA_ROOT` **before** importing config.


In [1]:
import os
import subprocess
import sys
from pathlib import Path

# A. Repository (code)
REPO_URL = "https://github.com/dk546/openf1-big-data-pipeline.git"
REPO_NAME = "openf1-big-data-pipeline"
PROJECT_ROOT = Path(f"/content/{REPO_NAME}")

# B. Google Drive persistence (outputs) — set OPENF1_DATA_ROOT before importing config
USE_GOOGLE_DRIVE = True
DRIVE_OUTPUT_ROOT = Path("/content/drive/MyDrive/openf1_big_data_pipeline")

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    os.environ["OPENF1_DATA_ROOT"] = str(DRIVE_OUTPUT_ROOT)
    print("OPENF1_DATA_ROOT:", os.environ["OPENF1_DATA_ROOT"])
else:
    os.environ.pop("OPENF1_DATA_ROOT", None)
    print("OPENF1_DATA_ROOT not set — repo-local outputs.")

# D. Clone or update repository
if not PROJECT_ROOT.exists():
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_ROOT)], check=True)
else:
    print("Updating repository...")
    subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull"], check=False)

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

# E. Verify project files
_checks = {
    "README.md": PROJECT_ROOT / "README.md",
    "pyproject.toml": PROJECT_ROOT / "pyproject.toml",
    "src/openf1_pipeline": PROJECT_ROOT / "src" / "openf1_pipeline",
    "src/openf1_pipeline/__init__.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "__init__.py",
    "src/openf1_pipeline/config.py": PROJECT_ROOT / "src" / "openf1_pipeline" / "config.py",
}
for name, path in _checks.items():
    if not path.exists():
        raise FileNotFoundError(f"Missing {name}: {path}")

# F. Install dependencies and editable package
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

# G. Fallback: src on sys.path
_src = PROJECT_ROOT / "src"
if str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# H. Import test
import openf1_pipeline  # noqa: E402

from openf1_pipeline.config import (  # noqa: E402
    ensure_project_directories,
    get_artifacts_dir,
    get_bronze_dir,
    get_data_dir,
    get_gold_dir,
    get_output_root,
    get_project_root,
    get_reports_dir,
    get_silver_dir,
)

paths = ensure_project_directories()

# I. Path summary
print("PROJECT_ROOT:", get_project_root())
print("OUTPUT_ROOT:", get_output_root())
print("DATA_DIR:", get_data_dir())
print("BRONZE_DIR:", get_bronze_dir())
print("SILVER_DIR:", get_silver_dir())
print("GOLD_DIR:", get_gold_dir())
print("REPORTS_DIR:", get_reports_dir())
print("ARTIFACTS_DIR:", get_artifacts_dir())



Mounted at /content/drive
OPENF1_DATA_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
Cloning repository...
Working directory: /content/openf1-big-data-pipeline
PROJECT_ROOT: /content/openf1-big-data-pipeline
OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
DATA_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data
BRONZE_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/bronze
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold
REPORTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/reports
ARTIFACTS_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/artifacts


## Start Spark


In [2]:
from openf1_pipeline.utils.spark import get_spark

spark = get_spark()
print("Spark version:", spark.version)


INFO:openf1_pipeline.utils.spark:Spark session started: openf1-big-data-pipeline


Spark version: 4.0.2


## Silver prerequisites


In [3]:
from pathlib import Path

import pandas as pd

from openf1_pipeline.config import (
    get_data_quality_reports_dir,
    get_feature_definitions_dir,
    get_gold_dir,
    get_output_root,
    get_silver_dir,
)
from openf1_pipeline.gold.build_feature_mart import build_gold_feature_mart
from openf1_pipeline.utils.io import read_parquet_if_exists
from openf1_pipeline.utils.cleanup import clean_gold_layer_outputs

GOLD_ENGINE = "spark"
ALLOW_FALLBACK = False
CLEAR_GOLD_OUTPUTS = True

SILVER_DIR = get_silver_dir()
GOLD_DIR = get_gold_dir()
DATA_QUALITY_REPORTS_DIR = get_data_quality_reports_dir()
FEATURE_DEFINITIONS_DIR = get_feature_definitions_dir()

print("OUTPUT_ROOT:", get_output_root())
print("SILVER_DIR:", SILVER_DIR)
print("GOLD_DIR:", GOLD_DIR)

required = [
    SILVER_DIR / "session_result_clean.parquet",
    SILVER_DIR / "laps_clean.parquet",
    SILVER_DIR / "drivers_clean.parquet",
]
for path in required:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing Silver table: {path}. Run 02_silver_cleaning_quality.ipynb first."
        )

starting_grid = SILVER_DIR / "starting_grid_clean.parquet"
if starting_grid.exists():
    sg = read_parquet_if_exists(starting_grid)
    if sg is not None and sg.empty:
        print("WARNING: starting_grid_clean.parquet is empty — grid features skipped.")
else:
    print("NOTE: starting_grid_clean.parquet not found (optional).")


OUTPUT_ROOT: /content/drive/MyDrive/openf1_big_data_pipeline
SILVER_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/silver
GOLD_DIR: /content/drive/MyDrive/openf1_big_data_pipeline/data/gold


## Clean Gold outputs (required before fresh Spark run)

Removes Gold Parquet, Gold DQ reports, DuckDB Gold reports, and feature dictionary.
Does not delete Silver.


In [4]:
if CLEAR_GOLD_OUTPUTS:
    print("Cleaning Gold layer outputs...")
    clean_gold_layer_outputs(
        gold_dir=GOLD_DIR,
        data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
        feature_definitions_dir=FEATURE_DEFINITIONS_DIR,
    )
else:
    print("Skipping Gold cleanup (CLEAR_GOLD_OUTPUTS=False).")


INFO:openf1_pipeline.utils.io:Cleaned directory contents /content/drive/MyDrive/openf1_big_data_pipeline/data/gold (1 items removed)
INFO:openf1_pipeline.utils.io:Removed 11 files matching ['gold_*.csv', 'duckdb_gold_*.csv'] from /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality
INFO:openf1_pipeline.utils.cleanup:Gold layer cleanup complete: {'gold_data': 1, 'gold_reports': 11, 'feature_dictionary': 1}


Cleaning Gold layer outputs...


## Silver pre-flight: defensive join-key dtype check

Before building Gold, confirm that join keys (`session_key`, `meeting_key`,
`driver_number`, `lap_number`, `position`) are stored as integer/long on
disk in every Silver Parquet. If a key is floating, joins still work in
Spark — but downstream DuckDB validation and per-key counts are clearer
when types are integral.

`build_race_control_features` intentionally ignores `race_control.qualifying_phase`
because that column is 100% null in current OpenF1 pulls (see the Silver
data-quality notes). Gold features use `message` and `flag` only.

If this cell prints a warning, rerun Notebook 02 with the latest
`clean_*_spark` modules (they now cast keys to long explicitly) and then
restart this notebook from the Silver pre-flight cell.

In [5]:
from openf1_pipeline.utils.spark import spark_path

KEY_COLUMNS_PER_TABLE = {
    "meetings": ["meeting_key", "year"],
    "sessions": ["session_key", "meeting_key", "year"],
    "drivers": ["session_key", "driver_number", "meeting_key"],
    "laps": ["session_key", "driver_number", "lap_number"],
    "pit": ["session_key", "driver_number", "lap_number"],
    "weather": ["session_key"],
    "position": ["session_key", "driver_number", "meeting_key", "position"],
    "race_control": ["session_key", "meeting_key"],
    "session_result": ["session_key", "driver_number", "meeting_key", "position"],
    "starting_grid": ["session_key", "driver_number", "position"],
}
INTEGRAL_DTYPES = {"int", "bigint", "smallint", "tinyint", "long"}

silver_schema_rows = []
for table_name, key_cols in KEY_COLUMNS_PER_TABLE.items():
    parquet_path = SILVER_DIR / f"{table_name}_clean.parquet"
    if not parquet_path.exists():
        silver_schema_rows.append({
            "table": table_name, "column": "(missing)", "dtype": "n/a", "is_integer": False,
        })
        continue
    sdf = spark.read.parquet(spark_path(parquet_path))
    schema = dict(sdf.dtypes)
    for col in key_cols:
        if col not in schema:
            continue
        dtype = schema[col]
        silver_schema_rows.append({
            "table": table_name, "column": col, "dtype": dtype,
            "is_integer": dtype.lower() in INTEGRAL_DTYPES,
        })

silver_schema_check = pd.DataFrame(silver_schema_rows)
display(silver_schema_check)

floating_keys = silver_schema_check.loc[
    ~silver_schema_check["is_integer"] & (silver_schema_check["dtype"] != "n/a")
]
if floating_keys.empty:
    print("OK: all Silver join keys are integer/long on disk — safe to build Gold.")
else:
    print("WARNING: the following Silver join keys are floating on disk —")
    display(floating_keys)
    print(
        "Rerun Notebook 02 with the latest clean_*_spark fixes (keys cast to long), "
        "then rerun this cell. Gold will still build, but join-key types should be "
        "integral for consistency with the data dictionary."
    )

,table,column,dtype,is_integer
0,meetings,meeting_key,bigint,True
1,meetings,year,bigint,True
2,sessions,session_key,bigint,True
3,sessions,meeting_key,bigint,True
4,sessions,year,bigint,True
5,drivers,session_key,bigint,True
6,drivers,driver_number,bigint,True
7,drivers,meeting_key,bigint,True
8,laps,session_key,bigint,True
9,laps,driver_number,bigint,True


OK: all Silver join keys are integer/long on disk — safe to build Gold.


## Build Gold feature mart (Spark-first)


In [6]:
outputs = build_gold_feature_mart(
    silver_dir=SILVER_DIR,
    gold_dir=GOLD_DIR,
    data_quality_reports_dir=DATA_QUALITY_REPORTS_DIR,
    feature_definitions_dir=FEATURE_DEFINITIONS_DIR,
    engine=GOLD_ENGINE,
    spark=spark,
    allow_fallback=ALLOW_FALLBACK,
)

outputs["summary"]


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/gold_feature_summary_stats.csv (49 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/gold_feature_missingness.csv (62 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/gold_target_distribution.csv (2 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/gold_join_quality_report.csv (6 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/gold_leakage_guard_report.csv (62 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/artifacts/feature_definitions/feature_dictionary.csv (62 rows)


{'engine': 'spark',
 'row_count': 1756,
 'column_count': 62,
 'model_feature_count': 40,
 'duplicate_grain_rows': 0,
 'points_finish_positive_pct': np.float64(47.4943)}

## DuckDB validation (Gold Parquet)


In [7]:
from openf1_pipeline.analytics.duckdb_validation import (
    save_duckdb_validation_reports,
    validate_gold_with_duckdb,
)

gold_duckdb = validate_gold_with_duckdb(outputs["paths"]["driver_race_feature_mart"])
duckdb_gold_paths = save_duckdb_validation_reports(
    gold_duckdb, DATA_QUALITY_REPORTS_DIR, prefix="gold"
)
duckdb_gold_paths


INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_row_count.csv (1 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report gold_row_count -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_row_count.csv (1 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_duplicate_keys.csv (0 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report gold_duplicate_keys -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_duplicate_keys.csv (0 rows)
INFO:openf1_pipeline.utils.io:Wrote CSV /content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_target_distribution.csv (2 rows)
INFO:openf1_pipeline.analytics.duckdb_validation:DuckDB report gold_target_distribution -> /content/drive/MyDrive/openf1_big_data_pipeline/reports/d

{'gold_row_count': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_row_count.csv',
 'gold_duplicate_keys': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_duplicate_keys.csv',
 'gold_target_distribution': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_target_distribution.csv',
 'points_finish_by_team': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_points_finish_by_team.csv',
 'points_finish_by_circuit': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_points_finish_by_circuit.csv',
 'gold_missingness_summary': '/content/drive/MyDrive/openf1_big_data_pipeline/reports/data_quality/duckdb_gold_gold_missingness_summary.csv'}

## Validate target distribution


In [8]:
target_dist = pd.read_csv(outputs["paths"]["gold_target_distribution"])
display(target_dist)


,points_finish,count,pct
0,0,922,52.5057
1,1,834,47.4943


## Validate feature missingness (top 20)


In [9]:
missingness = pd.read_csv(outputs["paths"]["gold_feature_missingness"])
display(
    missingness.sort_values("missing_pct", ascending=False).head(20)
)


,table_name,column_name,dtype,row_count,missing_count,missing_pct,non_missing_count,unique_count
12,driver_race_feature_mart,driver_country_code,object,1756,599,34.1116,1157,17
37,driver_race_feature_mart,avg_pit_duration,float64,1756,453,25.7973,1303,945
38,driver_race_feature_mart,min_pit_duration,float64,1756,453,25.7973,1303,687
39,driver_race_feature_mart,max_pit_duration,float64,1756,453,25.7973,1303,810
40,driver_race_feature_mart,first_pit_lap,float64,1756,453,25.7973,1303,56
7,driver_race_feature_mart,final_position,float64,1756,177,10.0797,1579,20
35,driver_race_feature_mart,std_first5_lap_duration,float64,1756,68,3.8724,1688,1688
30,driver_race_feature_mart,avg_i2_speed,float64,1756,58,3.3030,1698,1638
25,driver_race_feature_mart,std_lap_duration,float64,1756,49,2.7904,1707,1707
22,driver_race_feature_mart,avg_lap_duration,float64,1756,43,2.4487,1713,1713


## Validate join quality


In [10]:
join_quality = pd.read_csv(outputs["paths"]["gold_join_quality_report"])
display(join_quality)


,feature_group,rows_before,rows_after,unmatched_base_rows,unmatched_pct,notes
0,metadata,1756,1756,0,0.0,NaN
1,laps,1756,1756,0,0.0,NaN
2,pit,1756,1756,0,0.0,NaN
3,position,1756,1756,0,0.0,NaN
4,weather,1756,1756,0,0.0,NaN
5,race_control,1756,1756,0,0.0,NaN


## Leakage guard


In [11]:
leakage = pd.read_csv(outputs["paths"]["gold_leakage_guard_report"])
forbidden = leakage[leakage["allowed_for_modeling"] == False]
print(f"Columns blocked from modeling: {len(forbidden)}")
display(leakage[leakage["leakage_risk"].isin(["high", "target"])])
print(f"Model-safe feature count: {len(outputs['model_feature_columns'])}")


Columns blocked from modeling: 22


,column_name,leakage_risk,allowed_for_modeling,reason
3,result_dnf,high,False,post-race outcome or diagnostic field
4,result_dns,high,False,post-race outcome or diagnostic field
5,result_dsq,high,False,post-race outcome or diagnostic field
6,points_finish,target,False,model target
7,final_position,high,False,post-race outcome or diagnostic field
8,result_points,high,False,post-race outcome or diagnostic field
47,diagnostic_final_observed_position,high,False,post-race outcome or diagnostic field


Model-safe feature count: 40


## Feature dictionary


In [12]:
feature_dict = pd.read_csv(outputs["paths"]["feature_dictionary"])
display(feature_dict.head(20))
print("Predictive features:")
display(
    feature_dict[feature_dict["modeling_role"] == "predictive_feature"][
        ["feature_name", "feature_group", "missing_pct"]
    ].head(25)
)


,feature_name,feature_group,feature_tier,dtype,description,source_table,modeling_role,allowed_for_modeling,missing_pct
0,session_key,identifiers,none,int64,Grain key (identifiers).,session_result,identifier,False,0.0000
1,meeting_key,identifiers,none,int64,Grain key (identifiers).,session_result,identifier,False,0.0000
2,driver_number,identifiers,none,int64,Grain key (identifiers).,session_result,identifier,False,0.0000
3,result_dnf,diagnostic_outcome,none,bool,Post-race outcome for evaluation only; not for...,session_result,diagnostic_outcome,False,0.0000
4,result_dns,diagnostic_outcome,none,bool,Post-race outcome for evaluation only; not for...,session_result,diagnostic_outcome,False,0.0000
5,result_dsq,diagnostic_outcome,none,bool,Post-race outcome for evaluation only; not for...,session_result,diagnostic_outcome,False,0.0000
6,points_finish,target,none,int32,"Binary target: 1 if driver scored points (>0),...",session_result,target,False,0.0000
7,final_position,diagnostic_outcome,none,float64,Post-race outcome for evaluation only; not for...,session_result,diagnostic_outcome,False,10.0797
8,result_points,diagnostic_outcome,none,float64,Post-race outcome for evaluation only; not for...,session_result,diagnostic_outcome,False,0.0000
9,full_name,metadata,none,object,"Driver, session, or meeting metadata.","drivers,sessions,meetings",metadata,False,0.0000


Predictive features:


,feature_name,feature_group,missing_pct
21,lap_count,laps,0.0000
22,avg_lap_duration,laps,2.4487
23,median_lap_duration,laps,2.4487
24,best_lap_duration,laps,2.4487
25,std_lap_duration,laps,2.7904
26,avg_sector_1,laps,2.4487
27,avg_sector_2,laps,2.2210
28,avg_sector_3,laps,2.4487
29,avg_i1_speed,laps,1.8793
30,avg_i2_speed,laps,3.3030


## Inspect Gold output


In [13]:
gold_df = pd.read_parquet(outputs["paths"]["driver_race_feature_mart"])
print("Shape:", gold_df.shape)
print("Grain keys:", ["session_key", "meeting_key", "driver_number"])
display(gold_df.head(10))
print("Target rate (points_finish=1):", gold_df["points_finish"].mean())


Shape: (1756, 62)
Grain keys: ['session_key', 'meeting_key', 'driver_number']


,session_key,meeting_key,driver_number,result_dnf,result_dns,result_dsq,points_finish,final_position,result_points,full_name,...,avg_wind_speed,rainfall_mean,rainfall_flag,race_control_message_count,flag_message_count,yellow_flag_count,red_flag_count,green_flag_count,safety_car_message_count,pit_exit_message_count
0,7779,1142,1,False,False,False,0,2.0,0.0,Max VERSTAPPEN,...,1.772297,0.0,0,58,19,5,1,2,2,3
1,7779,1142,10,False,False,False,1,9.0,2.0,Pierre GASLY,...,1.772297,0.0,0,58,19,5,1,2,2,3
2,7779,1142,16,False,False,False,1,7.0,6.0,Charles LECLERC,...,1.772297,0.0,0,58,19,5,1,2,2,3
3,7779,1142,22,False,False,False,0,11.0,0.0,Yuki TSUNODA,...,1.772297,0.0,0,58,19,5,1,2,2,3
4,7779,1142,23,True,False,False,0,NaN,0.0,Alexander ALBON,...,1.772297,0.0,0,58,19,5,1,2,2,3
5,7779,1142,81,False,False,False,0,15.0,0.0,Oscar PIASTRI,...,1.772297,0.0,0,58,19,5,1,2,2,3
6,7787,1143,1,False,False,False,1,1.0,25.0,Max VERSTAPPEN,...,1.127027,0.0,0,111,49,12,5,2,9,3
7,7787,1143,2,True,False,False,0,16.0,0.0,Logan SARGEANT,...,1.127027,0.0,0,111,49,12,5,2,9,3
8,7787,1143,14,False,False,False,1,3.0,15.0,Fernando ALONSO,...,1.127027,0.0,0,111,49,12,5,2,9,3
9,7787,1143,16,True,False,False,0,NaN,0.0,Charles LECLERC,...,1.127027,0.0,0,111,49,12,5,2,9,3


Target rate (points_finish=1): 0.47494305239179957


## Checklist / next steps

- [ ] Copy Gold parquet + DQ CSVs to `evidence/<run_id>/` after smoke/full Colab run
- [ ] Confirm `gold_leakage_guard_report.csv` has no forbidden columns with `allowed_for_modeling=True`
- [ ] Proceed to **04 — modeling**
- [ ] Do **not** use `final_position`, `result_*`, or raw `position`/`points` as model inputs
